# Phase 0 - Explore the 1,000-persona sample

Loads and summarises a sample of `nvidia/Nemotron-Personas-Singapore` (synthetic Singaporean personas, CC BY 4.0). Data is fetched via the Hugging Face datasets-server HTTP API (no large downloads) and cached to CSV.

> Note: fully synthetic research data - no real individuals. Exploratory only.

In [1]:
import urllib.parse
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_CSV = DATA_DIR / "personas_sample_1000.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def build_rows_url(offset, length):
    params = {
        "dataset": "nvidia/Nemotron-Personas-Singapore",
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": length,
    }
    return "https://datasets-server.huggingface.co/rows?" + urllib.parse.urlencode(params)


def fetch_sample(max_rows=1000, page_size=100):
    """Fetch up to max_rows personas via the datasets-server API."""
    rows = []
    offset = 0
    while offset < max_rows:
        url = build_rows_url(offset, min(page_size, max_rows - offset))
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        page = [item["row"] for item in r.json()["rows"]]
        if not page:
            break
        rows.extend(page)
        offset += len(page)
    return pd.DataFrame(rows)


if SAMPLE_CSV.exists():
    df = pd.read_csv(SAMPLE_CSV)
    print(f"loaded {len(df):,} personas from cache")
else:
    df = fetch_sample(max_rows=1000)
    df.to_csv(SAMPLE_CSV, index=False)
    print(f"fetched and cached {len(df):,} personas")

loaded 1,000 personas from cache


In [3]:
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} cols\n")
info = df.dtypes.to_frame("dtype")
info["nulls"] = df.isna().sum()
print(info.to_string())

shape: 1,000 rows x 21 cols

                            dtype  nulls
uuid                          str      0
professional_persona          str      0
sports_persona                str      0
arts_persona                  str      0
travel_persona                str      0
culinary_persona              str      0
persona                       str      0
cultural_background           str      0
skills_and_expertise          str      0
skills_and_expertise_list     str      0
hobbies_and_interests         str      0
hobbies_and_interests_list    str      0
career_goals_and_ambitions    str      0
sex                           str      0
age                         int64      0
marital_status                str      0
education_level               str      0
occupation                    str      0
industry                      str    457
planning_area                 str      0
country                       str      0


In [4]:
print("--- age ---")
print(df["age"].describe().round(1).to_string())
for col in ["sex", "marital_status", "education_level"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().to_string())
print("\n--- planning_area (top 10) ---")
print(df["planning_area"].value_counts().head(10).to_string())

--- age ---
count    1000.0
mean       47.8
std        17.3
min        18.0
25%        33.0
50%        47.0
75%        61.0
max        96.0

--- sex ---
sex
Male      506
Female    494

--- marital_status ---
marital_status
Married               615
Single                284
Divorced/Separated     51
Widowed                50

--- education_level ---
education_level
University                       349
Secondary                        142
Post Secondary (Non-Tertiary)    134
Other Diploma                    105
Polytechnic                      102
Primary                           59
No Qualification                  59
Lower Secondary                   50

--- planning_area (top 10) ---
planning_area
Sengkang         71
Woodlands        64
Bedok            58
Jurong West      57
Tampines         57
Choa Chu Kang    55
Yishun           55
Pasir Ris        48
Punggol          47
Hougang          44


In [5]:
for i in range(3):
    row = df.iloc[i]
    print(f"--- persona {i + 1} (uuid {row['uuid'][:8]}) ---")
    print(row["persona"])
    print()

--- persona 1 (uuid d792702e) ---
Yi Peng Yong, known as Danelle, blends a love for meticulous budgeting, nightly journaling, and spontaneous karaoke renditions of Jay Chou, yet sometimes forgets to turn off her phone alarm, leading to early‑morning coffee runs.

--- persona 2 (uuid 2fe149f1) ---
Charmaine, a sociable 53‑year‑old office whiz with a karaoke‑loving streak, battles occasional anxiety, juggles church choir duties, weekend mahjong, and a penchant for perfecting mooncake crusts, all while eyeing a senior admin promotion.

--- persona 3 (uuid dd955147) ---
Betty is a methodical policy pro who balances her competitive edge with a quiet love for Sudoku, city‑scape photography, mindful meditation, and a secret obsession with collecting Jay Chou’s limited‑edition albums.



## Observations & next steps

- 21 columns; all populated except `industry` (blank for non-working personas).
- Ages 18-96, centred on working ages; balanced sex split.
- Next: LLM indicator injection (`src/inject_indicators.py`), then NLP feature
  extraction and detection evaluation. See `docs/PLAN.md`.